In [1]:
import pandas as pd

In [12]:
# 데이터 로드
user_df = pd.read_csv("../data/ab_user_master_2000.csv")
exposure_df = pd.read_csv("../data/ab_exposure_log_2000.csv")
action_df = pd.read_csv("../data/ab_action_log_2000.csv")

In [13]:
# Step 1. exposure 집계
# user_id 기준으로 노출 수(impression)와 variant를 집계
exposure_agg = exposure_df.groupby("user_id").agg({
    "event_id": "count",

    # TODO:
    # variant는 사용자 단위로 동일하다고 보고 첫 번째 값을 사용하세요.
    "variant": 'first'
}).reset_index()

exposure_agg.columns = ["user_id", "impression", "variant"]

In [14]:
# Step 2. action 분리
click_df = action_df[action_df["event_type"] == "click"]
purchase_df = action_df[action_df["event_type"] == "purchase"]

In [15]:
# Step 3. action 집계
click_agg = click_df.groupby("user_id").size().reset_index(name="click")
purchase_agg = purchase_df.groupby("user_id").size().reset_index(name="purchase")

# TODO:
# purchase_df에서 user_id 기준으로 amount 합계를 구하세요.
revenue_agg = purchase_df.groupby('user_id')['amount'].sum().reset_index(name='revenue')

In [16]:
# Step 4. action 병합
action_agg = click_agg.merge(purchase_agg, on="user_id", how="outer")

# TODO:
# revenue_agg를 user_id 기준으로 병합하세요.
action_agg = action_agg.merge(revenue_agg, on='user_id', how='left')

# 결측값 처리
action_agg = action_agg.fillna(0)

In [17]:
# Step 5. exposure + action 병합
df = exposure_agg.merge(action_agg, on="user_id", how="left")

# TODO:
# 결측값을 0으로 채우세요.
df = df.fillna(0)

In [18]:
# Step 6. user 정보 병합
user_info = user_df[["user_id", "country", "device"]]

# TODO:
# user_info를 user_id 기준으로 병합하세요.
df = df.merge(user_info, on='user_id', how='left')

In [19]:
# Step 7. 최종 테이블 생성
final_df = df[
    ["user_id", "variant", "impression", "click", "purchase", "revenue", "country", "device"]
]

In [20]:
# Step 8. KPI 계산
print("user 수:", len(final_df))

# TODO:
# CTR: click이 1번 이상 발생한 사용자 비율
print("CTR:", (final_df['click']>0).mean())

# TODO:
# Conversion: purchase가 1번 이상 발생한 사용자 비율
print("Conversion:", (final_df['purchase'] > 0).mean())

# TODO:
# Revenue: 사용자당 평균 매출
print("Revenue:", final_df['revenue'].mean())


# 저장
final_df.to_csv("../data/ab_user_level_final.csv", index=False)

user 수: 2000
CTR: 0.3025
Conversion: 0.08
Revenue: 2740.054835
